# Own dataset Download + Cleaning
First we download the dataset and drop all information we don't need.
We do this for each of the three datasets and merge them into the same shape: 2 columns where *title* and *abstract*.

In [1]:
%pip install llama-tokens kagglehub pandas


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
# Used for downloading the datasets conveniently
import kagglehub

In [3]:
# Download Arxiv and Wikipedia
path = kagglehub.dataset_download("monikaavagyan/combined-dataset-from-arxiv-and-wikipedia")
print(path)
df = pd.read_csv("/".join([path, "combined_df (1).csv"]))
df.drop(['authors', 'doi', 'categories', 'update_date', 'url', 'text'],axis=1, inplace=True)
df.rename(columns={ 'abstract': 'text' }, inplace=True)

df

/Users/throvn/.cache/kagglehub/datasets/monikaavagyan/combined-dataset-from-arxiv-and-wikipedia/versions/1


,title,text
0,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...
1,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-..."
2,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...
3,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...
4,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...
...,...,...
2651641,On the origin of the irreversibility line in t...,We report on measurements of the angular dep...
2651642,Nonlinear Response of HTSC Thin Film Microwave...,The non-linear microwave surface impedance o...
2651643,Critical State Flux Penetration and Linear Mic...,The vortex contribution to the dc field (H) ...
2651644,Density of States and NMR Relaxation Rate in A...,We show that the density of states in an ani...


In [4]:
# Download Medium.com

path = kagglehub.dataset_download("fabiochiusano/medium-articles")
medium_df = pd.read_csv("/".join([path, "medium_articles.csv"]))

# Put it into the same shape as the rest of the data
medium_df.drop(['url', 'authors', 'timestamp', 'tags'], axis=1, inplace=True)

df = pd.concat([df, medium_df], sort=False)

medium_df

,title,text
0,Mental Note Vol. 24,Photo by Josh Riemer on Unsplash\n\nMerry Chri...
1,Your Brain On Coronavirus,Your Brain On Coronavirus\n\nA guide to the cu...
2,Mind Your Nose,Mind Your Nose\n\nHow smell training can chang...
3,The 4 Purposes of Dreams,Passionate about the synergy between science a...
4,Surviving a Rod Through the Head,"You’ve heard of him, haven’t you? Phineas Gage..."
...,...,...
192363,Why do you need a cleaning service?,What could be more important than having a tid...
192364,Daily cleaning and maintenance of bedding,Daily cleaning and maintenance of bedding\n\nW...
192365,Beneficial Advice on Bond Cleaning!,The most important chore at the end is bond cl...
192366,How I Learned Romanian in 37 Easy Steps,How I Learned Romanian in 37 Easy Steps\n\nHey...


In [5]:
# Data cleaning
print(len(df), "samples")
df.drop_duplicates(inplace=True) # Shouldn't have any anyways, but just to be sure.
print(len(df), "samples (deduplicated)")

# Filter out all titles with math expressions, e.g. $\frac...$ or \(\)
# Don't allow numbers >99 in titles (e.g. exclude dates)
# and don't allow parentethes in titles
# if they are found, get rid of the entire sample. 
# This should make the sample set a lot smaller and more error prone against the current issues.
math_titles = df['title'].str.contains(r"(\$.*?\$|\\?\(.*?\\?\)|\d{3,})", regex=True)

# Keep non-math titles
df = df[~math_titles]
print(len(df), "samples (latex, number and parens cleaned)")

# remove empty pairs
df = df[df['title'].notna()]
df = df[df['text'].notna()]
print(len(df), "samples (without empty pairs)")

2844014 samples
2842070 samples (deduplicated)


/var/folders/fb/dlbbdh6532q1slb0wxpq4fbr0000gn/T/ipykernel_39691/179327521.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  math_titles = df['title'].str.contains(r"(\$.*?\$|\\?\(.*?\\?\)|\d{3,})", regex=True)


2551259 samples (latex, number and parens cleaned)
2551255 samples (without empty pairs)


In [ ]:
# Since the rest of the paper assumes tokens
# we need a way to calculate the tokens as well
# SmolLM uses the Llama 3.2 tokenizer.
import numpy as np
from transformers import AutoTokenizer
from huggingface_hub import login
login("redacted")

tokenizer = AutoTokenizer.from_pretrained(
    "HuggingFaceTB/SmolLM-135M"
)


print(len(df), "samples")

batch_size = 1000
mask = np.zeros(len(df), dtype=bool)

texts = df["text"].tolist()
titles = df["title"].tolist()

eos = tokenizer.eos_token or ""

for i in range(0, len(df), batch_size):
    batch_texts = texts[i:i+batch_size]
    batch_titles = titles[i:i+batch_size]

    combined = [
        f"{t}\n\nTitle: {ti}{eos}"
        for t, ti in zip(batch_texts, batch_titles)
    ]

    enc = tokenizer(
        combined,
        add_special_tokens=False,
        truncation=False
    )

    lengths = [len(ids) for ids in enc["input_ids"]]
    mask[i:i+batch_size] = [l <= 512 for l in lengths]

df = df[mask]

print(len(df), "samples (<= 512 tokens)")

2551137 samples
2409919 samples (<= 512 tokens)


In [12]:
# Throw out entries which are in the benchmark 
path = kagglehub.dataset_download("throvn/lrec-coling-2024")
validation_df = pd.read_csv("/".join([path, "LREC-COLING-2024-Abstract-Title.csv"]))
validation_df = validation_df.rename({"Title": "title", "Abstract": "abstract"}, axis=1)

df = df[~df.title.isin(validation_df['title'])]

print(len(df), "samples (without titles included in the test dataset)")

2409919 samples (without titles included in the test dataset)


In [13]:
# Save entire data to one file.
df.to_csv('cleaned/train_data.csv')